# AdaptiveTile Full Pipeline

This notebook runs the project end to end in the same order as the numbered scripts: data download, tile profiling, predictor training, preprocessing/EDA, retraining on preprocessed data, and benchmark experiments.

Run the cells top to bottom. The benchmark and training steps can take several minutes.

In [ ]:
from pathlib import Path
import runpy
import sys

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'src' / 'config.py').exists() and (candidate / 'scripts' / '01_download_data.py').exists():
            return candidate
    raise RuntimeError('Could not find the AdaptiveTile project root.')

project_root = find_project_root()
scripts_dir = project_root / 'scripts'
sys.path.insert(0, str(project_root))

print(f'Project root: {project_root}')
print(f'Python path updated with: {project_root}')

## 1. Download Data

This downloads Kodak and prints the manual steps for DIV2K and BSDS500.

In [ ]:
runpy.run_path(str(scripts_dir / '01_download_data.py'), run_name='__main__')

## 2. Profile Tiles

This creates `outputs/logs/tile_profiles.csv` from Kodak tiles.

In [ ]:
runpy.run_path(str(scripts_dir / '02_profile_tiles.py'), run_name='__main__')

## 3. Train Predictor

This trains the tabular and CNN predictors and saves `outputs/models/predictor.pkl`.

In [ ]:
runpy.run_path(str(scripts_dir / '03_train_predictor.py'), run_name='__main__')

## 4. EDA and Preprocessing

This generates the preprocessed tile dataset, EDA report, and plots.

In [ ]:
runpy.run_path(str(scripts_dir / '05_eda_preprocessing.py'), run_name='__main__')

## 5. Train on Preprocessed Data

This retrains the predictor on the preprocessed tile dataset and saves `outputs/models/predictor_preprocessed.pkl`.

In [ ]:
runpy.run_path(str(scripts_dir / '06_train_preprocessed.py'), run_name='__main__')

## 6. Run Benchmark Experiments

This runs all benchmark experiments and writes `outputs/logs/experiment_results.csv`.

In [ ]:
runpy.run_path(str(scripts_dir / '04_run_experiments.py'), run_name='__main__')

## 7. Quick Result Summary

This cell prints the generated CSV summaries so you can verify that the pipeline completed.

In [ ]:
import pandas as pd

logs_dir = project_root / 'outputs' / 'logs'
summary_files = [
    logs_dir / 'tile_profiles.csv',
    logs_dir / 'predictor_comparison.csv',
    logs_dir / 'tile_profiles_preprocessed.csv',
    logs_dir / 'predictor_comparison_preprocessed.csv',
    logs_dir / 'experiment_results.csv',
]

for path in summary_files:
    print(f'\n=== {path.name} ===')
    if path.exists():
        df = pd.read_csv(path)
        print(df.head().to_string(index=False))
        print(f'Rows: {len(df)}')
    else:
        print('Not found yet.')